In [0]:
%run ../setup/00_aws_connection

In [0]:
import pyspark.sql.functions as F


In [0]:
catalog_name = dbutils.widgets.get('catalog_name')
schema_bronze = dbutils.widgets.get('schema_bronze')

In [0]:
obj_buckets = s3.list_objects_v2(Bucket=name_bucket, Prefix='raw')

for obj in obj_buckets['Contents']:
    arquivo_key = obj['Key']

    if arquivo_key == 'raw/' or not arquivo_key.endswith('.csv'):
        continue

    name_file = arquivo_key.split('/')[-1]
    name_table = name_file.replace('.csv', '')

    raw_path = f's3://{name_bucket}/{arquivo_key}'
    bronze_path = f's3://{name_bucket}/bronze/{name_table}'

    table_catalog = f'{catalog_name}.{schema_bronze}.{name_table}'

    try:
        df = spark.read.format('csv') \
            .option('header', 'true') \
            .option('inferSchema', 'true') \
            .load(raw_path) \
            .withColumn('ingest_date', F.current_timestamp()) \
            .select('*', '_metadata.file_name', '_metadata.file_size', F.lit('aws_s3').alias('source'))
            
        df.write \
          .format("delta") \
          .mode("overwrite") \
          .option("path", bronze_path) \
          .option("delta.enableChangeDataFeed", "true") \
          .saveAsTable(table_catalog)

    except Exception as e:
        print(e)
